In [1]:
import pandas as pd
from datasets import load_dataset
from collections import Counter
from tqdm import tqdm

# --- 1. Load the Dataset ---
try:
    # Use streaming=True to avoid downloading the entire massive dataset
    dataset = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", streaming=True, split="guj_Gujr")
    print("✅ Successfully started loading the ai4bharat/IndicCorpV2 Gujarati dataset.")
except Exception as e:
    print(f"❌ Failed to load dataset. Error: {e}")
    exit()

# --- 2. Generate and Count N-grams ---
print("\nProcessing documents to generate n-grams...")

# Initialize counters to store n-gram frequencies
trigram_counts = Counter()
quadgram_counts = Counter()

# Set a limit for the number of documents to process from the stream.
# The full dataset is huge, so processing a subset is recommended.
# You can increase this number for a more comprehensive analysis.
NUM_DOCUMENTS_TO_PROCESS = 20000

# Use .take() to create a manageable subset of the streaming dataset
streaming_subset = dataset.take(NUM_DOCUMENTS_TO_PROCESS)

# Iterate through the documents with a progress bar
for document in tqdm(streaming_subset, total=NUM_DOCUMENTS_TO_PROCESS, desc="Analyzing text"):
    # Get the text content from the document
    text = document.get('text', '')
    if not text:
        continue

    # Simple tokenization by splitting on whitespace
    tokens = text.split()

    # Generate and count trigrams (sequences of 3 words)
    if len(tokens) >= 3:
        # Create a list of 3-word tuples: [('word1', 'word2', 'word3'), ('word2', 'word3', 'word4'), ...]
        trigrams = [tuple(tokens[i:i+3]) for i in range(len(tokens) - 2)]
        trigram_counts.update(trigrams)

    # Generate and count quadgrams (sequences of 4 words)
    if len(tokens) >= 4:
        # Create a list of 4-word tuples
        quadgrams = [tuple(tokens[i:i+4]) for i in range(len(tokens) - 3)]
        quadgram_counts.update(quadgrams)

print(f"\n✅ N-gram generation complete. Found {len(trigram_counts):,} unique trigrams and {len(quadgram_counts):,} unique quadgrams.")

# --- 3. Convert to DataFrame and Save to CSV ---
print("Converting n-gram counts to DataFrames and saving to CSV...")

# --- Process and Save Trigrams ---
try:
    # Convert the counter to a list of tuples and create a DataFrame
    # .most_common() sorts them by frequency in descending order
    trigram_df = pd.DataFrame(trigram_counts.most_common(), columns=['Ngram', 'Frequency'])

    # Join the tuple of words back into a single string for better readability in the CSV
    trigram_df['Ngram'] = trigram_df['Ngram'].apply(lambda x: ' '.join(x))

    # Save to a CSV file
    trigram_csv_path = 'trigrams.csv'
    # Use 'utf-8-sig' encoding to ensure Gujarati characters are displayed correctly in Excel
    trigram_df.to_csv(trigram_csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ Trigram data successfully saved to: {trigram_csv_path}")

except Exception as e:
    print(f"❌ Failed to save trigrams.csv. Error: {e}")


# --- Process and Save Quadgrams ---
try:
    # Convert the counter to a DataFrame, sorted by frequency
    quadgram_df = pd.DataFrame(quadgram_counts.most_common(), columns=['Ngram', 'Frequency'])

    # Join the tuple of words back into a single string
    quadgram_df['Ngram'] = quadgram_df['Ngram'].apply(lambda x: ' '.join(x))

    # Save to a CSV file
    quadgram_csv_path = 'quadgrams.csv'
    quadgram_df.to_csv(quadgram_csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ Quadgram data successfully saved to: {quadgram_csv_path}")

except Exception as e:
    print(f"❌ Failed to save quadgrams.csv. Error: {e}")

print("\n🚀 All tasks finished!")

/Users/kuldeepsinh/Desktop/LAB_III/NLP/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Successfully started loading the ai4bharat/IndicCorpV2 Gujarati dataset.

Processing documents to generate n-grams...


Analyzing text: 100%|██████████| 20000/20000 [00:07<00:00, 2514.51it/s]



✅ N-gram generation complete. Found 366,559 unique trigrams and 378,582 unique quadgrams.
Converting n-gram counts to DataFrames and saving to CSV...
✅ Trigram data successfully saved to: trigrams.csv
✅ Quadgram data successfully saved to: quadgrams.csv

🚀 All tasks finished!
